# Get ensemble transcriptions of the Daily Rainfall images

The daily rainfall ensemble transcriptions are a separate data source, stored
as a Parquet dataset for fast columnar processing with DuckDB.
Each JSON file holds keys Day 1 .. Day 31 plus a Totals block; every key maps
to 12 month slots (January-December) and each month slot has 5 ensemble
member values.

This section performs a full rebuild of that Parquet dataset from all JSON
files in the ensemble_transcriptions directory.

Run this notebook with the ADRQ kernel.

In [6]:
# Setup paths for ensemble JSON source and Parquet output.

import os
from pathlib import Path

import duckdb

from src.rainfall_rescue_sqlite.parquet_ingest import (
    default_ensemble_parquet_root,
    ingest_ensemble_to_parquet,
 )

repo_dir = f"{os.getenv('PDIR')}/Rainfall-Rescue"
rainfall_rescue_root = Path(repo_dir)

ensemble_root = "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions"
ensemble_root = Path(ensemble_root)

ensemble_dataset_root = default_ensemble_parquet_root()

print("RR root dir:", rainfall_rescue_root)
print("Ensemble root dir:", ensemble_root)
print("Ensemble dataset root:", ensemble_dataset_root)

RR root dir: /data/scratch/philip.brohan/ADRQ/Rainfall-Rescue
Ensemble root dir: /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions
Ensemble dataset root: /data/scratch/philip.brohan/ADRQ/ensemble_transcriptions_parquet


In [7]:
# Rebuild ensemble transcription Parquet dataset from scratch.
# Use max_files for a quick smoke run, then remove it for full ingest.

ensemble_result = ingest_ensemble_to_parquet(
    ensemble_root=ensemble_root,
    dataset_root=ensemble_dataset_root,
    max_files=50,
    overwrite=True,
)

ensemble_result

ParquetIngestionResult(dataset_root=PosixPath('/data/scratch/philip.brohan/ADRQ/ensemble_transcriptions_parquet'), source_root=PosixPath('/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions'), files_discovered=50, files_ingested=50, daily_rows=93000, total_rows=3000, errors=0)

## Full-scale ingest with SLURM

The in-process ingest above parses every JSON file in a single Python process,
which is fine for smoke tests but too slow for the full dataset. Because each
JSON file is self-contained, the work is embarrassingly parallel: we split the
(sorted) file list into shards and run them as a SLURM array job on the SPICE
cluster.

**How the work is sharded**

- The discovered JSON files are divided into ENSEMBLE_NUM_SHARDS (default 100)
  interleaved slices (files[shard_index::num_shards]). Each shard parses its
  own slice and writes a complete parquet shard dataset under
  ENSEMBLE_SHARD_DIR/ens_shard_XXXXX/.
- A final merge step reads all shard datasets, re-numbers file_id values to keep
  foreign keys consistent across tables, and writes one consolidated dataset to
  ENSEMBLE_PARQUET_ROOT (default: $PDIR/ensemble_transcriptions_parquet).

**Pipeline (two stages, chained with --dependency=afterok)**

| Stage | Script | SLURM | Purpose |
|-------|--------|-------|---------|
| ingest | scripts/slurm/ingest_ensemble_array.sbatch | array 0-99 | parse each shard's JSON slice in parallel |
| merge  | scripts/slurm/merge_ensemble_shards.sbatch | 1 job | combine shards into one parquet dataset |

Shard count, resources and optional file cap are set in
scripts/slurm/config.sh (ENSEMBLE_NUM_SHARDS, ENSEMBLE_MAX_FILES, and the
EINGEST_*/EMERGE_* resource variables).

**Setting the source directory.** The ensemble_root variable set in the notebook
cell above is not seen by SLURM jobs directly. Set ENSEMBLE_ROOT when launching
the submit script; it is passed explicitly to each job via --ensemble-root.

**Launch the whole pipeline** (from the repository root, on a login node):

```bash
ENSEMBLE_ROOT=/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions \
scripts/slurm/submit_ensemble_ingest.sh
```

Quick test on a small slice (500 files across the shards):

```bash
ENSEMBLE_ROOT=/path/to/ensemble_transcriptions \
ENSEMBLE_MAX_FILES=500 scripts/slurm/submit_ensemble_ingest.sh
```

**Monitor it**

```bash
squeue -u $USER                                              # queued / running
sacct --format=JobID%15,JobName%14,State,ExitCode,Elapsed   # final states
ls -t $PDIR/slurm_logs | head                               # per-job logs
```

With 100 shards the serial parse time drops to a few minutes per shard plus
a merge step. The cells below inspect the resulting parquet dataset.

In [9]:
# Basic check of ensemble Parquet dataset contents.

conn = duckdb.connect()
try:
    files_n = conn.execute(
        f"SELECT COUNT(*) FROM read_parquet('{ensemble_dataset_root / 'ensemble_files' / '*.parquet'}')"
    ).fetchone()[0]
    daily_n = conn.execute(
        f"SELECT COUNT(*) FROM read_parquet('{ensemble_dataset_root / 'ensemble_daily_values' / '*.parquet'}')"
    ).fetchone()[0]
    totals_n = conn.execute(
        f"SELECT COUNT(*) FROM read_parquet('{ensemble_dataset_root / 'ensemble_monthly_totals' / '*.parquet'}')"
    ).fetchone()[0]
    
    # Errors table may be empty if ingest had zero errors
    try:
        errors_n = conn.execute(
            f"SELECT COUNT(*) FROM read_parquet('{ensemble_dataset_root / 'ingestion_file_errors' / '*.parquet'}')"
        ).fetchone()[0]
    except Exception:
        errors_n = 0
finally:
    conn.close()

print("Latest ingest summary")
print(f"  Files ingested: {files_n}")
print(f"  Daily rows: {daily_n}")
print(f"  Total rows: {totals_n}")
print(f"  Errors: {errors_n}")

Latest ingest summary
  Files ingested: 584513
  Daily rows: 1087194180
  Total rows: 35070780
  Errors: 0


In [10]:
# View any parse errors from the Parquet ingest.

conn = duckdb.connect()
try:
    # Errors table may be empty if ingest had zero errors
    try:
        errors = conn.execute(
            f"""
            SELECT source_path, error_message
            FROM read_parquet('{ensemble_dataset_root / 'ingestion_file_errors' / '*.parquet'}')
            ORDER BY source_path
            LIMIT 20
            """
        ).fetchall()
    except Exception:
        errors = []
finally:
    conn.close()

if errors:
    print(f"Found {len(errors)} parse errors (showing up to 20):")
    for source_path, error_message in errors:
        print(f"  {source_path}")
        print(f"    -> {error_message}\n")
else:
    print("No errors recorded in ingestion_file_errors")

No errors recorded in ingestion_file_errors


In [11]:
# Sample daily ensemble values for one file (day 1, all months and members).

conn = duckdb.connect()
try:
    file_row = conn.execute(
        f"""
        SELECT file_id, file_name, year_start, year_end, descriptor, section_id, num_days
        FROM read_parquet('{ensemble_dataset_root / 'ensemble_files' / '*.parquet'}')
        ORDER BY file_name
        LIMIT 1
        """
    ).fetchone()

    print(
        f"file_id={file_row[0]}  {file_row[1]}  "
        f"years={file_row[2]}-{file_row[3]}  "
        f"descriptor={file_row[4]}  section={file_row[5]}  "
        f"days={file_row[6]}"
    )

    daily_sample = conn.execute(
        f"""
        SELECT month, ensemble_member, rainfall
        FROM read_parquet('{ensemble_dataset_root / 'ensemble_daily_values' / '*.parquet'}')
        WHERE file_id = ? AND day_of_month = 1
        ORDER BY month, ensemble_member
        """
        , [file_row[0]]
    ).fetchall()
finally:
    conn.close()

for month, ensemble_member, rainfall in daily_sample[:15]:
    print(f"  month={month:>2}  member={ensemble_member}  rainfall={rainfall}")

file_id=1  DRain_1861-1870_Alderney-0.json  years=None-None  descriptor=None  section=None  days=31
  month= 1  member=1  rainfall=None
  month= 1  member=2  rainfall=None
  month= 1  member=3  rainfall=None
  month= 1  member=4  rainfall=None
  month= 1  member=5  rainfall=None
  month= 2  member=1  rainfall=None
  month= 2  member=2  rainfall=None
  month= 2  member=3  rainfall=None
  month= 2  member=4  rainfall=None
  month= 2  member=5  rainfall=None
  month= 3  member=1  rainfall=None
  month= 3  member=2  rainfall=None
  month= 3  member=3  rainfall=None
  month= 3  member=4  rainfall=None
  month= 3  member=5  rainfall=None


In [12]:
# Monthly totals for the same file: mean ensemble total per month.

conn = duckdb.connect()
try:
    totals = conn.execute(
        f"""
        SELECT month,
               AVG(total) AS mean_total,
               COUNT(total) AS n_values
        FROM read_parquet('{ensemble_dataset_root / 'ensemble_monthly_totals' / '*.parquet'}')
        WHERE file_id = ?
        GROUP BY month
        ORDER BY month
        """
        , [file_row[0]]
    ).fetchall()
finally:
    conn.close()

for month, mean_total, n_values in totals:
    mean_str = f"{mean_total:.3f}" if mean_total is not None else "n/a"
    print(f"  month={month:>2}  mean_total={mean_str:>8}  members_with_value={n_values}")

  month= 1  mean_total=     n/a  members_with_value=0
  month= 2  mean_total=     n/a  members_with_value=0
  month= 3  mean_total=     n/a  members_with_value=0
  month= 4  mean_total=     n/a  members_with_value=0
  month= 5  mean_total=     n/a  members_with_value=0
  month= 6  mean_total=     n/a  members_with_value=0
  month= 7  mean_total=     n/a  members_with_value=0
  month= 8  mean_total=     n/a  members_with_value=0
  month= 9  mean_total=     n/a  members_with_value=0
  month=10  mean_total=     n/a  members_with_value=0
  month=11  mean_total=     n/a  members_with_value=0
  month=12  mean_total=     n/a  members_with_value=0
